In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler


In [ ]:
df = pd.read_csv(r"C:\Users\josel\Downloads\Edtechfinal.csv")
df = df.reset_index().rename(columns={"index": "app_id"})


In [3]:
df["score"] = pd.to_numeric(df["score"], errors="coerce").fillna(0)
df["ratings"] = pd.to_numeric(df["ratings"], errors="coerce").fillna(0)
df["reviews"] = pd.to_numeric(df["reviews"], errors="coerce").fillna(0)
df["installs"] = pd.to_numeric(df["installs"], errors="coerce").fillna(0)
df["updated"] = pd.to_numeric(df["updated"], errors="coerce").fillna(0)


In [4]:
def size_to_mb(size):
    if isinstance(size, str):
        if "M" in size:
            return float(size.replace("M", "").strip())
        if "k" in size.lower():
            return float(size.lower().replace("k", "").strip()) / 1024
    return np.nan

df["size_mb"] = df["size"].apply(size_to_mb)
df["size_mb"] = df["size_mb"].fillna(df["size_mb"].median())


In [5]:
df["score_norm"] = df["score"] / 5

df["ratings_norm"] = np.log1p(df["ratings"])
df["reviews_norm"] = np.log1p(df["reviews"])
df["installs_norm"] = np.log1p(df["installs"])

scaler = MinMaxScaler()
df[["ratings_norm", "reviews_norm", "installs_norm", "size_norm"]] = scaler.fit_transform(
    df[["ratings_norm", "reviews_norm", "installs_norm", "size_mb"]]
)


In [6]:
now = datetime.now().timestamp()
df["update_recency"] = 1 - (now - df["updated"]) / (now - df["updated"].min())
df["update_recency"] = df["update_recency"].clip(0, 1)


In [7]:
def android_score(v):
    if isinstance(v, str) and "Varies" in v:
        return 1.0
    try:
        return min(float(v) / 10, 1)
    except:
        return 0.5

df["android_norm"] = df["androidVersion"].apply(android_score)


In [8]:
df["ads_penalty"] = df["containsAds"].apply(lambda x: 0.05 if x else 0)


In [9]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=50
)

tfidf_matrix = tfidf.fit_transform(df["title"])
df["CK"] = tfidf_matrix.mean(axis=1).A1
df["CK"] = MinMaxScaler().fit_transform(df[["CK"]])


In [10]:
df["TK"] = df[["size_norm", "update_recency", "android_norm"]].mean(axis=1)


In [11]:
df["PK"] = df[["score_norm", "ratings_norm"]].mean(axis=1) - df["ads_penalty"]
df["PK"] = df["PK"].clip(0, 1)


In [12]:
df["TPK"] = df[["TK", "PK", "reviews_norm"]].mean(axis=1)


In [13]:
df["TCK"] = df[["TK", "CK"]].mean(axis=1)
df["PCK"] = df[["PK", "CK"]].mean(axis=1)


In [14]:
df["final_score"] = (
    0.20 * df["TK"] +
    0.25 * df["CK"] +
    0.25 * df["PK"] +
    0.15 * df["TPK"] +
    0.10 * df["TCK"] +
    0.05 * df["PCK"]
)


In [15]:
final_df = df[[
    "app_id",
    "title",
    "TK",
    "CK",
    "PK",
    "TPK",
    "TCK",
    "PCK",
    "final_score"
]]


In [16]:
final_df.to_csv("edtech_tpack_recommender.csv", index=False)
print("✅ Dataset generado: edtech_tpack_recommender.csv")


✅ Dataset generado: edtech_tpack_recommender.csv


In [ ]:
final_df.head(30)


,app_id,title,TK,CK,PK,TPK,TCK,PCK,final_score
0,0,BYJU'S – The Learning App,0.575924,0.539642,0.851937,0.647852,0.557783,0.695789,0.650825
1,1,Duolingo: language lessons,0.586937,0.381666,0.907380,0.831439,0.484302,0.644523,0.645021
2,2,"IIT JEE, NEET, NCERT Solutions",0.406189,0.539595,0.827739,0.555013,0.472892,0.683667,0.587796
3,3,Unacademy Learner App,0.586654,0.515007,0.757029,0.601273,0.550830,0.636018,0.612414
4,4,Noon Academy – Student Learning App,0.441490,0.738761,0.778440,0.560407,0.590125,0.758600,0.648602
5,5,Cambly - English Teacher,0.495492,0.381666,0.776566,0.604839,0.438579,0.579116,0.552196
6,6,Edmodo,0.415787,0.000000,0.744277,0.628325,0.207893,0.372138,0.402872
7,7,Quizlet: Languages & Vocab,0.587038,0.000000,0.816671,0.739245,0.293519,0.408336,0.482231
8,8,Udemy - Online Courses,0.450902,0.635197,0.826599,0.663635,0.543050,0.730898,0.646025
9,9,Teachmint - The Classroom App,0.432177,0.524811,0.758089,0.478464,0.478494,0.641450,0.558852


: 